# Sweeping an input value

A parameter sweep (notebook 02) changes a tuned constant the flow reads with
`param()`. An input sweep changes a record's own value: `income` here, a
column of the input. An `overrides` without `at` writes an input before
anything runs; overriding a value a step produces needs `at=<step path>`.

In [1]:
import polars as pl
from decider.experiments import (Bars, Changes, Describe, Divergence, Experiment, Progress, RunStatus,
                                 ValueCounts)
from example_projects.experiments.credit_pipeline import build

ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [250.0, 500.0, 700.0, 250.0],
})
df

income,debt
f64,f64
1000.0,250.0
2000.0,500.0
1500.0,700.0
800.0,250.0


In [2]:
exp = Experiment("income-sweep", pipeline, df, entry=ENTRY)
for income in (600.0, 900.0, 1200.0, 3000.0):
    exp.scenario(f"income_{income:.0f}", overrides={"income": income})
exp.scenario("first_record_rich", overrides={"income": 10_000.0}, row=0)
exp.scenario("ratio_forced", at="credit/ratio", overrides={"ratio": 0.1})
exp.metric(Divergence(column="approved")).plot(Changes(metric="divergence"))

result = exp.run()
{name: (s.changed, s.first) for name, s in result.summaries["divergence"].items()}

{'baseline': (0, None),
 'income_600': (2, 0),
 'income_900': (2, 1),
 'income_1200': (2, 1),
 'income_3000': (2, 2),
 'first_record_rich': (0, None),
 'ratio_forced': (2, 2)}

In [3]:
result.plots["changes"].show()

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'changed': 0, 'compared': 4, 'first': None}, {'scenario': 'income_600', 'changed': 2, 'compared': 4, 'first': 0}, {'scenario': 'income_900', 'changed': 2, 'compared': 4, 'first': 1}, {'scenario': 'income_1200', 'changed': 2, 'compared': 4, 'first': 1}, {'scenario': 'income_3000', 'changed': 2, 'compared': 4, 'first': 2}, {'scenario': 'first_record_rich', 'changed': 0, 'compared': 4, 'first': None}, {'scenario': 'ratio_forced', 'changed': 2, 'compared': 4, 'first': 2}]}, 'mark': {'type': 'bar'}, 'encoding': {'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'changed', 'type': 'quantitative'}, {'field': 'compared', 'type': 'quantitative'}, {'field': 'first', 'title': 'first changed record', 'type': 'quantitative'}], 'x': {'field': 'scenario', 'sort': None, 'type': 'nominal'}, 'y': {'field': 'changed', 'title': 'records changed', 'type': 'quantitative'}}, 'title': 'approved changed vs baseline', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})

`validate` returns every problem with a definition at once, before anything
runs: an override of a produced value with no `at`, a plot naming a metric
the experiment doesn't collect, or one whose metric gives the wrong kind of
summary.

In [4]:
from decider.experiments import validate

broken = (Experiment("broken", pipeline, df)
          .scenario("no_point", overrides={"ratio": 0.1})
          .metric(ValueCounts(column="approved"))
          .plot(Changes(metric="value_counts")))
validate(broken.def_, pipeline)

["no_point: override 'ratio' is produced, not an input; declare 'at: <step path>'",
 "plot 'changes' needs a Divergences summary; metric 'value_counts' gives Counts"]